# L'escalier des taux : montée éclair, descente prudente · *The rate staircase: lightning climb, cautious descent*

Notebook compagnon du chapitre **8. La carte du voyage : de la COVID au soft landing (2020-2025)** — [lire l'article](https://nmlab.io/ressources/de-la-covid-au-soft-landing-2020-2025).
Companion notebook to chapter **8. The Map of the Journey: From COVID to the Soft Landing (2020-2025)** — [read the article](https://nmlab.io/en/ressources/from-covid-to-the-soft-landing-2020-2025).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_rates() -> tuple[Series, Series]:
    """Taux directeurs quotidiens : borne haute de la Fed (DFEDTARU) et taux de la
    facilité de dépôt de la BCE (ECBDFR), 2019-2025, en direct de FRED.
    Daily policy rates: Fed target top (DFEDTARU) and ECB deposit rate (ECBDFR)."""
    fed = nm.load_fred("DFEDTARU", start="2019").loc["2019":"2025"].dropna()
    ecb = nm.load_fred("ECBDFR", start="2019").loc["2019":"2025"].dropna()
    return fed, ecb

fed, ecb = load_rates()


from pandas import Timestamp as T
import matplotlib.dates as mdates
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="L'escalier des taux : montée éclair, descente prudente",
        legend=["Fed — borne haute de la fourchette", "BCE — taux de la facilité de dépôt"],
        fed_zero="mars 2020 : 0-0,25 %\nen deux réunions d'urgence",
        ecb_exit="juil. 2022 : la BCE sort\nde huit ans de taux négatifs",
        fed_peak="juil. 2023 : 5,25-5,50 %",
        ecb_peak="sept. 2023 :\ndépôt à 4,00 %",
        cuts="premières baisses :\nBCE juin, Fed sept. 2024",
        end_fed="3,50-3,75 %", end_ecb="2,00 %",
        note="Sources : Réserve fédérale (borne haute des fed funds) et BCE (facilité de dépôt), via FRED, quotidien.\n"
             "Fed : +525 pb en 16 mois, BCE : +450 pb en 14 mois. Les dates sont les prises d'effet des décisions."),
    "en": dict(
        title="The rate staircase: lightning climb, cautious descent",
        legend=["Fed — top of the target range", "ECB — deposit facility rate"],
        fed_zero="Mar. 2020: 0-0.25%\nin two emergency meetings",
        ecb_exit="Jul. 2022: the ECB exits\neight years of negative rates",
        fed_peak="Jul. 2023: 5.25-5.50%",
        ecb_peak="Sept. 2023:\ndeposit at 4.00%",
        cuts="first cuts:\nECB June, Fed Sept. 2024",
        end_fed="3.50-3.75%", end_ecb="2.00%",
        note="Sources: Federal Reserve (top of the fed funds range) and ECB (deposit facility), via FRED, daily.\n"
             "Fed: +525 bp in 16 months, ECB: +450 bp in 14 months. Dates are the effective dates of the decisions."),
}

def build_figure(fed: Series, ecb: Series, lang: str) -> Figure:
    """Deux escaliers de taux, réunions et premières baisses annotées."""
    text = LABELS[lang]
    pct = FuncFormatter(lambda v, _: (f"{v:.0f} %" if lang == "fr" else f"{v:.0f}%").replace("-", "−"))
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.062, top=0.86)
    ax.axhline(0, color=nm.COLORS["muted"], linewidth=1.3, alpha=0.6)
    ax.plot(fed.index, fed, color=nm.COLORS["blue"], linewidth=2.6, label=text["legend"][0])
    ax.plot(ecb.index, ecb, color=nm.COLORS["rose"], linewidth=2.6, label=text["legend"][1])

    fed_cut = fed[fed.diff() < 0].loc["2024"].index[0]
    ecb_cut = ecb[ecb.diff() < 0].loc["2024"].index[0]
    ax.scatter([fed_cut], [fed.loc[fed_cut]], s=130, facecolors="none",
               edgecolors=nm.COLORS["blue"], linewidths=2.4, zorder=6)
    ax.scatter([ecb_cut], [ecb.loc[ecb_cut]], s=130, facecolors="none",
               edgecolors=nm.COLORS["rose"], linewidths=2.4, zorder=6)

    ax.set_ylim(-1.35, 6.4)
    ax.set_yticks(range(-1, 7))
    ax.yaxis.set_major_formatter(pct)
    ax.set_xlim(T("2019-01-01"), T("2026-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.annotate(text["fed_zero"], xy=(T("2020-03-16"), 0.25), xytext=(T("2020-06-01"), 1.55),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["ecb_exit"], xy=(T("2022-07-27"), -0.5), xytext=(T("2020-08-01"), -0.75),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.text(T("2022-08-01"), 5.85, text["fed_peak"], fontsize=21, color=nm.COLORS["text"], va="center", ha="left")
    ax.text(T("2023-07-01"), 3.15, text["ecb_peak"], fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5)
    ax.text(T("2023-10-01"), 1.55, text["cuts"], fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5)

    ax.text(T("2024-08-01"), 3.35, text["end_fed"], fontsize=24, fontweight="bold",
            color=nm.COLORS["blue2"], va="center", ha="left")
    ax.text(T("2025-03-01"), 1.75, text["end_ecb"], fontsize=24, fontweight="bold",
            color=nm.COLORS["rose"], va="center", ha="left")

    leg = ax.legend(loc="upper left", bbox_to_anchor=(0.015, 0.97), fontsize=21,
                    labelcolor=nm.COLORS["text"], handlelength=1.5, borderpad=0.8,
                    labelspacing=0.6, fancybox=True)
    leg.get_frame().set_facecolor(nm.COLORS["card"])
    leg.get_frame().set_edgecolor(nm.COLORS["edge"])
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(fed, ecb, LANG)